# Aggregating User Preferences while Ensuring Equity, Diversity and Inclusion
## Évaluation des baselines — MovieLens 100k

**Adji Marieme Sita Cissé** — Stage M2 DataScale, Université Paris-Saclay  
Supervision : Prof. Malek Mouhoub — University of Regina  
Juillet 2026

---
Ce notebook reproduit les expériences de la Section 5 du papier.  
Il calcule les métriques EDI (équité ΔE, diversité ILD, inclusion) pour trois méthodes de référence sur MovieLens 100k.

In [ ]:
import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import combinations

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

DATA_DIR = os.path.join('data', 'ml-100k')
COLORS = {'Average Score': '#4C72B0', 'Borda': '#DD8452', 'Condorcet': '#55A868'}
K     = 10
THETA = 4.0

## 1. Chargement et exploration des données

MovieLens 100k (GroupLens Research) : 943 utilisateurs, 1 682 films, 100 000 notes de 1 à 5.  
Attribut sensible retenu : **genre** (F / M), directement disponible dans `u.user`.

In [ ]:
def load_movielens():
    ratings = pd.read_csv(
        os.path.join(DATA_DIR, 'u.data'),
        sep='\t', header=None,
        names=['user_id', 'item_id', 'rating', 'timestamp']
    )
    users = pd.read_csv(
        os.path.join(DATA_DIR, 'u.user'),
        sep='|', header=None,
        names=['user_id', 'age', 'gender', 'occupation', 'zip']
    )
    return ratings, users

ratings, users = load_movielens()
gender_map = users.set_index('user_id')['gender'].to_dict()

n_F     = sum(1 for g in gender_map.values() if g == 'F')
n_M     = sum(1 for g in gender_map.values() if g == 'M')
alpha_F = n_F / len(gender_map)
alpha_M = n_M / len(gender_map)

print(f'Ratings  : {len(ratings):,}')
print(f'Users    : {len(gender_map)} total  =>  {n_F} femmes ({alpha_F:.0%})  |  {n_M} hommes ({alpha_M:.0%})')
print(f'Items    : {ratings["item_id"].nunique()}')
print(f'Densite  : {len(ratings) / (len(gender_map) * ratings["item_id"].nunique()):.1%}')
print(f'Note moy.: {ratings["rating"].mean():.2f}  (ecart-type {ratings["rating"].std():.2f})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Distribution des notes
note_counts = ratings['rating'].value_counts().sort_index()
axes[0].bar(note_counts.index, note_counts.values, color='#4C72B0', edgecolor='white')
axes[0].set_title('Distribution des notes')
axes[0].set_xlabel('Note')
axes[0].set_ylabel('Nombre de ratings')
axes[0].set_xticks([1, 2, 3, 4, 5])

# Repartition par genre
bars = axes[1].bar(['Femmes (F)', 'Hommes (M)'], [n_F, n_M],
                   color=['#DD8452', '#4C72B0'], edgecolor='white')
axes[1].set_title('Repartition par genre')
axes[1].set_ylabel('Nombre d\'utilisateurs')
for bar, v, pct in zip(bars, [n_F, n_M], [alpha_F, alpha_M]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{v}\n({pct:.0%})', ha='center', fontweight='bold', fontsize=10)

# Ratings par utilisateur
ratings_per_user = ratings.groupby('user_id').size()
axes[2].hist(ratings_per_user, bins=30, color='#55A868', edgecolor='white')
axes[2].axvline(ratings_per_user.mean(), color='red', linestyle='--', label=f'Moy. {ratings_per_user.mean():.0f}')
axes[2].set_title('Ratings par utilisateur')
axes[2].set_xlabel('Nombre de ratings')
axes[2].set_ylabel('Frequence')
axes[2].legend()

plt.suptitle('MovieLens 100k — Vue d\'ensemble', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_data_overview.png', bbox_inches='tight')
plt.show()
print('Figure sauvegardee : fig_data_overview.png')

## 2. Graphe biparti G = (U ∪ I, E, w, s)

Le modele de preference est un graphe biparti pondere et attribue :

| Symbole | Definition |
|---|---|
| U | Noeuds utilisateurs (943) |
| I | Noeuds items / films (1 682) |
| E ⊆ U × I | Aretes — (u, i) ∈ E si u a note i |
| w : E → [1, 5] | Poids = note donnee par u a i |
| s : U → {F, M} | Attribut sensible = genre |

> **Propriete bipartite :** aucune arete entre deux utilisateurs ni entre deux items.

In [ ]:
def build_graph(ratings, users):
    G = nx.Graph()
    gmap = users.set_index('user_id')['gender'].to_dict()
    for uid, gender in gmap.items():
        G.add_node(f'u{uid}', bipartite=0, gender=gender)
    for _, row in ratings.iterrows():
        uid, iid, w = int(row['user_id']), int(row['item_id']), float(row['rating'])
        G.add_node(f'i{iid}', bipartite=1)
        G.add_edge(f'u{uid}', f'i{iid}', weight=w)
    return G, gmap

print('Construction du graphe...')
G, _ = build_graph(ratings, users)

n_user_nodes = sum(1 for _, d in G.nodes(data=True) if d.get('bipartite') == 0)
n_item_nodes = sum(1 for _, d in G.nodes(data=True) if d.get('bipartite') == 1)
weights = [d['weight'] for _, _, d in G.edges(data=True)]

print(f'Noeuds : {G.number_of_nodes()} ({n_user_nodes} utilisateurs + {n_item_nodes} items)')
print(f'Aretes : {G.number_of_edges():,}')
print(f'Poids  : min={min(weights):.0f}  max={max(weights):.0f}  moy={np.mean(weights):.2f}')

## 3. Méthodes de référence (baselines)

Trois methodes classiques d'agregation de preferences :

| Methode | Principe | Limite EDI connue |
|---|---|---|
| **Average Score** | score(i) = moyenne des notes | Favorise items populaires dans le groupe majoritaire |
| **Borda** | Chaque utilisateur classe ses films, on somme les rangs | Biais quand les groupes sont de tailles inegales |
| **Condorcet** | Item i bat j si > 50% des utilisateurs preferent i | La majorite peut systématiquement ecraser la minorite |

In [ ]:
def top_k_average_score(ratings, k=10):
    scores = ratings.groupby('item_id')['rating'].mean()
    return list(scores.nlargest(k).index)


def top_k_borda(ratings, k=10):
    def borda_scores(group):
        ranked = group.sort_values('rating', ascending=False).reset_index(drop=True)
        ranked['borda'] = range(len(ranked), 0, -1)
        return ranked[['item_id', 'borda']]
    borda = ratings.groupby('user_id', group_keys=False).apply(borda_scores)
    scores = borda.groupby('item_id')['borda'].sum()
    return list(scores.nlargest(k).index)


def top_k_condorcet(ratings, k=10):
    total_wins = {}
    for _, group in ratings.groupby('user_id'):
        items  = group['item_id'].values
        scores = group['rating'].values
        if len(items) < 2:
            continue
        wins_per_item = np.sum(scores[:, None] > scores[None, :], axis=1)
        for i, item_id in enumerate(items):
            total_wins[item_id] = total_wins.get(item_id, 0) + int(wins_per_item[i])
    ranked = sorted(total_wins.items(), key=lambda x: x[1], reverse=True)
    return [item for item, _ in ranked[:k]]


print('Calcul des top-10 (k=10)...')
top_avg  = top_k_average_score(ratings, K)
top_bord = top_k_borda(ratings, K)
print('  Average Score et Borda : OK')
print('  Condorcet en cours (peut prendre ~30 secondes)...')
top_cond = top_k_condorcet(ratings, K)
print('  Condorcet : OK')

print(f'\nTop-10 Average Score : {top_avg}')
print(f'Top-10 Borda         : {top_bord}')
print(f'Top-10 Condorcet     : {top_cond}')

## 4. Métriques EDI

Trois metriques pour evaluer une liste de recommandation R = (i₁, …, i_k) :

**E — Equite (Yao & Huang, 2017)**  
`ΔE = |utility_F(R) − utility_M(R)|`  où  `utility_g(R) = (1/|G_g|) Σ_{u∈G_g} Σ_{i∈R} w(u,i) / k`

**D — Diversite**  
`ILD(R) = (2 / k(k−1)) Σ_{i≠j ∈ R} dist(i, j)`  où  `dist(i,j) = 1 − cosine_sim(v_i, v_j)`

**I — Inclusion (Kellerhals & Peters, 2024)**  
`inclusion_g(R) = (1/|G_g|) Σ_{u∈G_g} |{i ∈ R : w(u,i) ≥ θ}| / k`  
Cible : `inclusion_g ≥ α_g` (part proportionnelle du groupe)

In [ ]:
def group_utility(ratings, top_k_items, gender_map):
    top_k_set = set(top_k_items)
    k = len(top_k_items)
    results = {}
    for gender in ['F', 'M']:
        group_users = [uid for uid, g in gender_map.items() if g == gender]
        gr = ratings[ratings['user_id'].isin(group_users) & ratings['item_id'].isin(top_k_set)]
        results[gender] = gr['rating'].sum() / (len(group_users) * k) if group_users and k else 0.0
    return results


def delta_E(ratings, top_k_items, gender_map):
    util = group_utility(ratings, top_k_items, gender_map)
    return abs(util['F'] - util['M'])


def ILD(ratings, top_k_items):
    k = len(top_k_items)
    if k < 2:
        return 0.0
    pivot = ratings.pivot(index='user_id', columns='item_id', values='rating').fillna(0)
    vectors = {iid: pivot[iid].values if iid in pivot.columns else np.zeros(len(pivot))
               for iid in top_k_items}
    total_dist, n_pairs = 0.0, 0
    for i, j in combinations(top_k_items, 2):
        vi, vj = vectors[i], vectors[j]
        norm = np.linalg.norm(vi) * np.linalg.norm(vj)
        total_dist += 1 - (np.dot(vi, vj) / norm if norm > 0 else 0.0)
        n_pairs += 1
    return total_dist / n_pairs if n_pairs > 0 else 0.0


def inclusion(ratings, top_k_items, gender_map, theta=4.0):
    top_k_set = set(top_k_items)
    k = len(top_k_items)
    results = {}
    for gender in ['F', 'M']:
        group_users = [uid for uid, g in gender_map.items() if g == gender]
        if not group_users or k == 0:
            results[gender] = 0.0
            continue
        gr = ratings[
            ratings['user_id'].isin(group_users) &
            ratings['item_id'].isin(top_k_set) &
            (ratings['rating'] >= theta)
        ]
        rel_per_user = gr.groupby('user_id')['item_id'].nunique()
        total = rel_per_user.reindex(group_users, fill_value=0).sum()
        results[gender] = total / (len(group_users) * k)
    return results


print('Fonctions EDI definies.')

## 5. Évaluation des baselines

In [ ]:
def evaluate_all(ratings, gender_map, baselines_dict, theta=4.0):
    rows = []
    for name, top_k in baselines_dict.items():
        dE  = delta_E(ratings, top_k, gender_map)
        ild = ILD(ratings, top_k)
        inc = inclusion(ratings, top_k, gender_map, theta)
        rows.append({'Methode': name, 'dE': dE, 'ILD': ild,
                     'inc_F': inc['F'], 'inc_M': inc['M']})
    return pd.DataFrame(rows).set_index('Methode')


baselines = {
    'Average Score': top_avg,
    'Borda':         top_bord,
    'Condorcet':     top_cond,
}

results = evaluate_all(ratings, gender_map, baselines, THETA)

print(f'k={K}, theta={THETA}  |  alpha_F={alpha_F:.2f}, alpha_M={alpha_M:.2f}\n')
print(results.to_string(float_format=lambda x: f'{x:.4f}'))

In [ ]:
methods = results.index.tolist()
bar_colors = [COLORS[m] for m in methods]
x = np.arange(len(methods))
w = 0.5

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# --- delta_E ---
axes[0].bar(x, results['dE'], width=w, color=bar_colors, edgecolor='white')
axes[0].axhline(0, color='red', linewidth=1.5, linestyle='--', label='Cible : 0')
axes[0].set_title('Equite  (\u0394E)\n[plus bas = mieux]')
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods, rotation=15, ha='right')
axes[0].set_ylim(0, 0.55)
axes[0].legend()
for xi, v in zip(x, results['dE']):
    axes[0].text(xi, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

# --- ILD ---
axes[1].bar(x, results['ILD'], width=w, color=bar_colors, edgecolor='white')
axes[1].set_title('Diversite  (ILD)\n[plus haut = mieux]')
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods, rotation=15, ha='right')
axes[1].set_ylim(0, 1.15)
for xi, v in zip(x, results['ILD']):
    axes[1].text(xi, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)

# --- Inclusion ---
xF = x - 0.2
xM = x + 0.2
axes[2].bar(xF, results['inc_F'], width=0.35, color='#DD8452', label='Femmes (F)', edgecolor='white')
axes[2].bar(xM, results['inc_M'], width=0.35, color='#4C72B0', label='Hommes (M)', edgecolor='white')
axes[2].axhline(alpha_F, color='#DD8452', linestyle='--', linewidth=1.5, label=f'alpha_F = {alpha_F:.2f}')
axes[2].axhline(alpha_M, color='#4C72B0', linestyle='--', linewidth=1.5, label=f'alpha_M = {alpha_M:.2f}')
axes[2].set_title('Inclusion\n[plus haut = mieux, cible = alpha_g]')
axes[2].set_xticks(x)
axes[2].set_xticklabels(methods, rotation=15, ha='right')
axes[2].set_ylim(0, 1.0)
axes[2].legend(fontsize=8)

plt.suptitle(f'Metriques EDI des baselines  (k={K}, theta={THETA})',
             fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('fig_edi_baselines.png', bbox_inches='tight')
plt.show()
print('Figure sauvegardee : fig_edi_baselines.png')

## 6. Synthèse et interprétation

### Ce que montrent les résultats

| Methode | Forces | Faiblesses |
|---|---|---|
| **Average Score** | Equite quasi-parfaite (ΔE ≈ 0), diversite maximale (ILD = 1.0) | Inclusion quasi-nulle pour les deux groupes : les items a note moyenne la plus haute sont des films tres peu notes — personne ne les a vus |
| **Borda** | Inclusion feminine au seuil (≈ α_F = 0.29) | Equite faible (ΔE = 0.42), inclusion masculine tres insuffisante (0.39 vs cible 0.71) |
| **Condorcet** | Resultats similaires a Borda, legere avance sur ILD | Memes problemes d'equite et d'inclusion masculine |

### Conclusion

> **Aucune baseline ne satisfait simultanement ΔE ≈ 0, ILD eleve, et les deux contraintes d'inclusion.**  
> Ce resultat confirme la motivation de notre approche : une **coarsenisation du graphe EDI-contrainte** peut-elle trouver un meilleur equilibre ?

La ligne *Ours* du tableau de la Section 5 sera remplie apres implementation de l'algorithme de coarsenisation.

In [ ]:
print('=== TABLE LATEX (Section 5) ===')
print()
print(r'\begin{tabular}{lcccc}')
print(r'\hline')
print(r'Method & $\Delta E \downarrow$ & ILD $\uparrow$ & $\mathrm{inc}_F \uparrow$ & $\mathrm{inc}_M \uparrow$ \\')
print(r'\hline')
for method, row in results.iterrows():
    best_dE  = (row['dE']  == results['dE'].min())
    best_ild = (row['ILD'] == results['ILD'].max())
    best_iF  = (row['inc_F'] == results['inc_F'].max())
    best_iM  = (row['inc_M'] == results['inc_M'].max())

    def fmt(v, best):
        return f'\\textbf{{{v:.3f}}}' if best else f'{v:.3f}'

    print(f'{method:<14} & {fmt(row["dE"],best_dE)} & {fmt(row["ILD"],best_ild)} & '
          f'{fmt(row["inc_F"],best_iF)} & {fmt(row["inc_M"],best_iM)} \\\\')

print('Ours           &                &                &                &       \\\\')
print(r'\hline')
print(f'$\\alpha_g$ cible & -- & -- & {alpha_F:.2f} & {alpha_M:.2f} \\\\')
print(r'\hline')
print(r'\end{tabular}')